# HelioAI — the 2003 Halloween storm

One of the largest geomagnetic storms of the space age, worked end to end: find the
parameters, download them, characterise the shock and the driver, and compute the plasma
regimes on either side.

This notebook is a **worked scientific example** rather than a feature tour — see
`01_jupyter_tour.ipynb` for that.

**Before running this**

1. `pip install helioai` (or `uv sync` from a clone)
2. One LLM provider key in `.env` — see the
   [installation guide](https://erdoganfurkan.github.io/HelioAI/installation/)
3. `helioai index` once (~10 min)

> Outputs are stripped on purpose — run the cells to produce your own. Wall time is
> dominated by data download; expect a few minutes.


In [ ]:
%load_ext helioai.interfaces.jupyter_magic

import os
provider = os.environ.get("HELIOAI_LLM_PROVIDER", "azure")
print(f"LLM provider : {provider}")
print("Ready — run cells below with Shift+Enter.")

---
## Act I — Data Discovery

Before any analysis, you need correct speasy parameter IDs. The same physical quantity appears under dozens of names across ACE, Wind, DSCOVR, STEREO…  
Classic pitfalls: `onboard` vs `prime` moments, `RTN` vs `GSE` frame, `HIA` vs `CODIF` for ions, `MFI` vs `MAG` for magnetometers.

The `parameter_hunter` sub-agent searches the 83 000-entry RAG index and returns the canonical IDs with caveats.

In [ ]:
%%helioai
I want to analyze the Halloween geomagnetic storm driven by the X17.2
CME from AR 10486. The event window is 2003-10-28T20:00:00 to
2003-10-30T06:00:00 UT.

I need four ACE quantities:
  1. IMF vector in GSM (or GSE) at the highest available cadence
  2. Proton number density
  3. Solar wind bulk speed
  4. Proton temperature

And the same four quantities from Wind, so I can compare shock arrival
times between the two spacecraft later.

For each parameter give me the exact speasy ID, the native cadence,
and any data-quality caveats (saturation, gaps, frame) I should know
about for this particular event.

---
## Act II — Solar Wind Visualization

The shock arrived at ACE at approximately **06:11 UT on October 29**.
The sheath that followed had large-amplitude IMF fluctuations before the prolonged southward Bz interval that drove the main storm phase.

**Reference values to check your output against:**

| Quantity | Expected | Source |
|:---|:---|:---|
| Shock arrival at ACE | ~06:11 UT Oct 29 | Wind MFI Shock Catalog |
| Pre-shock solar wind speed | ~400–450 km/s | ACE SWEPAM |
| Post-shock peak speed | ~1850–2000 km/s | ACE SWEPAM |
| Peak sheath density | ~70–100 cm⁻³ | ACE SWEPAM |
| Min sheath IMF Bz (GSM) | ~−35 nT | ACE MFI |
| Main southward Bz onset | ~14:00–15:00 UT Oct 29 | ACE MFI |

> **Note:** The speed may be clipped in SWEPAM data (instrument designed for ≤ 1200 km/s nominal). The agent should flag this.

In [ ]:
%%helioai
Using the ACE parameter IDs you just identified, download and plot
the Halloween storm solar wind for 2003-10-28T20:00:00 to
2003-10-30T06:00:00 UT.

Make a four-panel time series figure (dark background, shared x-axis):
  Panel 1 -- IMF |B| (black) and Bz in GSM (blue when > 0, red when < 0)
             draw a thin grey dashed line at Bz = 0
  Panel 2 -- Proton number density (cm^-3)
  Panel 3 -- Solar wind bulk speed (km/s)
  Panel 4 -- Proton temperature (eV or K, your choice)

On each panel:
  - Mark the shock arrival with a vertical orange line -- find it
    from the data rather than assuming 06:11 UT
  - Shade the sheath region (between shock and the first sustained
    southward-Bz interval) in semi-transparent orange
  - Mark the onset of the main southward-Bz driving interval with a
    dashed vertical red line labelled 'storm main phase onset'

After the figure, print a summary table with:
  - Detected shock time (UT)
  - Upstream (1-hour average before shock): |B|, n, V, T
  - Downstream (30-min average after shock): |B|, n, V, T
  - Min Bz in the sheath
  - Whether the speed looks clipped (> 1200 km/s in SWEPAM)

---
## Act III — Rankine-Hugoniot Shock Physics

The Rankine-Hugoniot jump conditions for a fast-mode MHD shock constrain the compression ratio, Mach number, and downstream temperature.  
For a strong shock in a γ = 5/3 gas, the theoretical maximum compression is:

$$r_{\max} = \frac{\gamma+1}{\gamma-1} = 4$$

The Alfvénic Mach number $M_A = V_{\mathrm{shock}} / V_A^{\mathrm{upstream}}$ classifies shock strength — for the Halloween event, $M_A$ was exceptionally high, making it one of the most efficient particle accelerators observed at L1.

**Expected:**

| Quantity | Expected | Implication |
|:---|:---|:---|
| Density compression r | 3.5–4 | Near strong-shock limit |
| B compression | 3–4 | Field amplification in sheath |
| V_A upstream | ~30–45 km/s | Low-β solar wind |
| M_A | > 15 | Very strong shock |
| Downstream temperature | Much hotter | Rankine-Hugoniot heating |

In [ ]:
%%helioai
Using the upstream/downstream averages you computed at the Halloween
storm shock, perform a Rankine-Hugoniot analysis:

1. Density compression ratio  r_n = n2 / n1
2. Magnetic field compression r_B = |B|2 / |B|1
3. Upstream Alfven speed V_A1 = |B|1 / sqrt(mu0 * m_p * n1)
4. Estimate the shock speed V_shock using the de Hoffmann-Teller
   approximation: V_shock = V2 * r_n / (r_n - 1)
5. Alfvenic Mach number M_A = V_shock / V_A1
6. Compare r_n to the theoretical strong-shock limit (gamma+1)/(gamma-1)
   for gamma = 5/3 -- is this near the strong-shock limit?
7. Compute the expected downstream temperature from Rankine-Hugoniot:
   T2 = T1 * [2*gamma*Ms^2 - (gamma-1)] * [(gamma-1)*Ms^2 + 2]
                / [(gamma+1)^2 * Ms^2]
   where Ms is the sonic Mach number (use c_s1 = sqrt(gamma*k_B*T1/m_p))
   and compare to the observed T2

Conclude with a one-paragraph physical interpretation: shock strength,
implications for solar energetic particle acceleration via diffusive
shock acceleration (DSA), and whether the observed compression is
consistent with an MHD description or hints at non-MHD effects.

---
## Act IV — Multi-spacecraft Shock Normal

The Halloween shock hit L1 spacecraft in sequence. ACE and Wind were separated by several tens of R_E in the Y_GSE direction.  
The arrival time difference $\Delta t$ between spacecraft pairs, combined with their separation vector $\mathbf{\Delta r}$, constrains the shock normal:

$$\hat{n} \parallel \frac{\mathbf{\Delta r}_{12}}{V_{\text{shock}} \cdot \Delta t_{12}}$$

For the Halloween event, the shock normal was close to the Sun–Earth line ($n_x \approx -1$) with a small $Y_{\rm GSE}$ component, consistent with a nearly radial propagation from AR 10486 at S17E08.

**Reference (Berdichevsky et al. 2005):** shock normal ~(−0.99, 0.08, 0.04) in GSE.

In [ ]:
%%helioai
Compare the Halloween storm shock arrival at ACE and Wind.
Time window: 2003-10-29T05:30:00 to 2003-10-29T08:00:00 UT.

Fetch IMF |B| and proton density from both spacecraft at native
cadence and overlay them on two plots (B and n side by side,
different colours per spacecraft, legend showing names).

Then:
1. Cross-correlate the density time series to measure the timing
   lag delta_t (ACE ahead or behind Wind?)
2. Given approximate GSE positions at shock arrival:
     ACE  : (232,  41,  21) R_E
     Wind : (  3, -85,  18) R_E
   Compute the separation vector dR = R_Wind - R_ACE (in km).
3. Using your shock speed estimate from Act III and delta_t,
   estimate the shock normal direction in GSE.
4. Compare to the Berdichevsky et al. 2005 reference:
   n_hat = (-0.99, 0.08, 0.04) GSE.
   Is the propagation close to radial? What does the Y-component
   tell us about the CME source location (S17E08 on the solar disk)?

---
## Act V — Instant Plasma Physics Reference

For quick sanity checks, call PlasmaPy tools **directly** — zero latency, no LLM call, no API key needed.  
Useful before an agent session to define expected value ranges, or after to verify the agent's numbers.

In [ ]:
from helioai.tools.plasmapy_tools import (
    plasma_beta, alfven_speed, gyrofrequency, debye_length, inertial_length
)

# Halloween 2003-10-29 -- three key plasma regimes
regions = {
    "Quiet SW upstream    (Oct 28 23 UT)": {"B_nT":  7.0, "n_cm3":  6.0, "T_eV": 12.0},
    "Shocked sheath        (Oct 29 07 UT)": {"B_nT": 27.0, "n_cm3": 26.0, "T_eV": 70.0},
    "Southward-Bz driver  (Oct 29 20 UT)": {"B_nT": 32.0, "n_cm3": 18.0, "T_eV": 40.0},
}

hdr = f"{'Region':<38}  {'beta':>5}  {'VA km/s':>9}  {'fci Hz':>8}  {'lD m':>7}  {'di km':>7}"
print(hdr)
print("-" * len(hdr))

results = {}
for label, p in regions.items():
    b   = await plasma_beta(p["B_nT"], p["n_cm3"], p["T_eV"])
    va  = await alfven_speed(p["B_nT"], p["n_cm3"])
    fci = await gyrofrequency(p["B_nT"], "proton")
    ld  = await debye_length(p["n_cm3"], p["T_eV"])
    di  = await inertial_length(p["n_cm3"], "proton")
    results[label] = {"b": b, "va": va}
    print(
        f"{label:<38}  {b['beta']:>5.2f}  "
        f"{va['alfven_speed_km_s']:>9.1f}  "
        f"{fci['frequency_Hz']:>8.4f}  "
        f"{ld['debye_length_m']:>7.2f}  "
        f"{di['inertial_length_km']:>7.1f}"
    )

print()
ups  = regions["Quiet SW upstream    (Oct 28 23 UT)"]
down = regions["Shocked sheath        (Oct 29 07 UT)"]
va1  = results["Quiet SW upstream    (Oct 28 23 UT)"]["va"]["alfven_speed_km_s"]
print(f"Density compression r_n  : {down['n_cm3'] / ups['n_cm3']:.1f}x")
print(f"B-field compression r_B  : {down['B_nT']  / ups['B_nT']  :.1f}x")
print(f"Upstream V_A             : {va1:.1f} km/s")
print(f"Strong-shock limit (g=5/3): r_max = 4.0")

---
## Act VI — Reproducible Export

Every `run_python` call from the agent is **automatically saved** as a `code_N.py` script alongside the figures in the workspace.  
Ask the agent to bundle the full session into a standalone script — runnable with only `speasy`, `numpy`, `matplotlib`, and `plasmapy`. No HelioAI, no API key.

This is the artefact you attach to a paper or share with a colleague.

In [ ]:
%%helioai
Export the complete Halloween storm analysis from this session as a
single standalone Python script.

The script should:
  1. Download ACE + Wind data using speasy (no HelioAI import)
  2. Reproduce the four-panel solar wind plot with shock annotation
  3. Print the Rankine-Hugoniot table
  4. Print the shock normal estimate
  5. Have a single CONFIG block at the top (date range, parameter IDs,
     figure output path) so any user can adapt it to a different event
     by changing only that block

Add a header comment with: event name, references (Skoug et al. 2004 JGR
and Berdichevsky et al. 2005), and the speasy version pinned.
Save the script as halloween_2003_standalone.py in the workspace.

---
## What you just did — and what took 3 hours before HelioAI

| Task | Old workflow | HelioAI |
|:---|:---|:---|
| Find correct speasy IDs | 30 min CDAWeb browsing | ~30 s |
| Write download + plot code | 1–2 h Python/IDL | ~60 s |
| Compute Rankine-Hugoniot | 30 min manual formulas | ~60 s |
| Multi-spacecraft timing | 1 h coordinate geometry | ~90 s |
| Export reproducible script | 30 min refactoring | ~30 s |
| **Total** | **3–5 h** | **< 10 min** |

The agent made **zero assumptions** about parameter IDs — it searched the 83 000-entry catalog and returned the canonical speasy paths with caveats specific to this event (SWEPAM saturation above 1200 km/s, frame conventions).

---

**References**
- Skoug et al. (2004), *JGR*, 109, A09102 — ACE solar wind during Halloween storms
- Berdichevsky et al. (2005), *Solar Phys.*, 227, 173 — Shock normal analysis
- Mewaldt et al. (2005), *JGR*, 110, A09S18 — SEP fluences and DSA
- Gopalswamy et al. (2005), *JGR*, 110, A09S15 — CME kinematics

**Links**
- [HelioAI on GitHub](https://github.com/erdoganfurkan/HelioAI)
- [speasy documentation](https://speasy.readthedocs.io)
- [PlasmaPy documentation](https://docs.plasmapy.org)
- [ACE Science Center](https://www.srl.caltech.edu/ACE/ASC/)
- [Wind MFI IP Shock Catalog](https://wind.nasa.gov/mfi/ip_shock.html)